In [8]:
import pandas as pd
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, precision_score, f1_score, recall_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

## Get songs in Spotify data format (e.g. danceability, energy, valence, ...)

In [41]:
#df = pd.read_csv("../data/raw/1200_song_mapped.csv",",")
df = pd.read_csv('../data/raw/full_survey_w_all_spotify_data.csv',',')
df.head()

,Unnamed: 0,index,Response_Id,Timestamp,Current_Mood,Recent_Songs,Recent_Movies,Music_Genre,Movie_Genre,Mood_to_Entrtnmnt,...,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature,popularity
0,0,0,0,9/28/2023 17:40:17,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,...,0.803,114.995,audio_features,2YXGbxICUdOUJe9OPlicy1,spotify:track:2YXGbxICUdOUJe9OPlicy1,https://api.spotify.com/v1/tracks/2YXGbxICUdOU...,https://api.spotify.com/v1/audio-analysis/2YXG...,143833,4,76
1,1,1,0,9/28/2023 17:40:17,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,...,0.642,115.042,audio_features,1xzi1Jcr7mEi9K2RfzLOqS,spotify:track:1xzi1Jcr7mEi9K2RfzLOqS,https://api.spotify.com/v1/tracks/1xzi1Jcr7mEi...,https://api.spotify.com/v1/audio-analysis/1xzi...,225389,4,86
2,2,2,0,9/28/2023 17:40:17,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,...,0.748,129.979,audio_features,3IX0yuEVvDbnqUwMBB3ouC,spotify:track:3IX0yuEVvDbnqUwMBB3ouC,https://api.spotify.com/v1/tracks/3IX0yuEVvDbn...,https://api.spotify.com/v1/audio-analysis/3IX0...,184784,4,94
3,3,3,0,9/28/2023 17:40:17,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,...,0.350,138.005,audio_features,1kuGVB7EU95pJObxwvfwKS,spotify:track:1kuGVB7EU95pJObxwvfwKS,https://api.spotify.com/v1/tracks/1kuGVB7EU95p...,https://api.spotify.com/v1/audio-analysis/1kuG...,219724,4,96
4,4,4,0,9/28/2023 17:40:17,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,...,0.775,110.056,audio_features,1vYXt7VSjH9JIM5oRRo7vA,spotify:track:1vYXt7VSjH9JIM5oRRo7vA,https://api.spotify.com/v1/tracks/1vYXt7VSjH9J...,https://api.spotify.com/v1/audio-analysis/1vYX...,176579,4,95


In [42]:
# Drop unnecessary columns
df = df.drop(['Unnamed: 0', 'index','Timestamp'],axis=1)

In [43]:
df.columns

Index(['Response_Id', 'Current_Mood', 'Recent_Songs', 'Recent_Movies',
       'Music_Genre', 'Movie_Genre', 'Mood_to_Entrtnmnt', 'Music_to_Movie',
       'Artist', 'Track', 'audio_features', 'track_info', 'danceability',
       'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'type', 'id', 'uri',
       'track_href', 'analysis_url', 'duration_ms', 'time_signature',
       'popularity'],
      dtype='object')

In [44]:
# Raw data that will be fed into the robust scaler and then model
X = df[['danceability','acousticness','energy','instrumentalness','liveness','valence','loudness','speechiness','tempo']]
X

,danceability,acousticness,energy,instrumentalness,liveness,valence,loudness,speechiness,tempo
0,0.861,0.07640,0.611,0.000000,0.0763,0.8030,-7.891,0.0427,114.995
1,0.780,0.03680,0.689,0.000010,0.0698,0.6420,-5.668,0.1410,115.042
2,0.627,0.00193,0.879,0.000007,0.0647,0.7480,-3.446,0.0955,129.979
3,0.511,0.17700,0.532,0.000000,0.2910,0.3500,-5.745,0.0578,138.005
4,0.671,0.02070,0.845,0.000000,0.3290,0.7750,-4.930,0.0480,110.056
...,...,...,...,...,...,...,...,...,...
805,0.858,0.17400,0.471,0.000004,0.1750,0.2730,-9.725,0.1650,99.981
806,0.538,0.00302,0.556,0.877000,0.1060,0.0357,-9.779,0.0349,94.974
807,0.640,0.00696,0.511,0.000536,0.1450,0.1540,-9.194,0.0363,120.014
808,0.555,0.56000,0.450,0.000000,0.3110,0.7060,-13.436,0.0421,146.834


## Preprocess songs' features for model

In [45]:
# Load robust scaler
loaded_rb = pickle.load(open("rb_scaler.pkl", "rb"))

In [46]:
# Transform the features
X_transformed = loaded_rb.transform(X)
X_transformed

array([[ 1.40856844, -0.15150411,  0.04015444, ...,  0.03050293,
         0.00374532, -0.15636044],
       [ 1.07001045, -0.19591242,  0.16061776, ...,  0.25101676,
         3.68539326, -0.15495877],
       [ 0.43051202, -0.2350164 ,  0.45405405, ...,  0.47143141,
         1.98127341,  0.29050296],
       ...,
       [ 0.48484848, -0.22937565, -0.11428571, ..., -0.09875012,
        -0.23595506, -0.00668029],
       [ 0.12957158,  0.39081555, -0.20849421, ..., -0.51954171,
        -0.01872659,  0.79316464],
       [ 1.06165099, -0.14903698,  0.15598456, ..., -0.18306716,
         1.43820225,  0.29056261]])

## Run model on songs and append back to dataframe

In [47]:
# Load model
filename = "music_to_mood_model.pickle"
loaded_model = pickle.load(open(filename, "rb"))

In [48]:
# Get predictions on data
y_pred = loaded_model.predict(X_transformed)

In [52]:
# Mapping emotions
emotions_mapping = {0: 'sad', 1: 'happy', 2:'energetic', 3:'calm'}

In [53]:
# Concatenate mood predictions to main dataframe
df['mood'] = y_pred
df['mood'] = df['mood'].map(emotions_mapping)
df

,Response_Id,Current_Mood,Recent_Songs,Recent_Movies,Music_Genre,Movie_Genre,Mood_to_Entrtnmnt,Music_to_Movie,Artist,Track,...,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature,popularity,mood
0,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Lizzo,Pink,...,114.995,audio_features,2YXGbxICUdOUJe9OPlicy1,spotify:track:2YXGbxICUdOUJe9OPlicy1,https://api.spotify.com/v1/tracks/2YXGbxICUdOU...,https://api.spotify.com/v1/audio-analysis/2YXG...,143833,4,76,happy
1,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Beyonce,Cuff It,...,115.042,audio_features,1xzi1Jcr7mEi9K2RfzLOqS,spotify:track:1xzi1Jcr7mEi9K2RfzLOqS,https://api.spotify.com/v1/tracks/1xzi1Jcr7mEi...,https://api.spotify.com/v1/audio-analysis/1xzi...,225389,4,86,happy
2,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Olivia Rodrigo,Bad Idea Right,...,129.979,audio_features,3IX0yuEVvDbnqUwMBB3ouC,spotify:track:3IX0yuEVvDbnqUwMBB3ouC,https://api.spotify.com/v1/tracks/3IX0yuEVvDbn...,https://api.spotify.com/v1/audio-analysis/3IX0...,184784,4,94,energetic
3,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Olivia Rodrigo,Vampire,...,138.005,audio_features,1kuGVB7EU95pJObxwvfwKS,spotify:track:1kuGVB7EU95pJObxwvfwKS,https://api.spotify.com/v1/tracks/1kuGVB7EU95p...,https://api.spotify.com/v1/audio-analysis/1kuG...,219724,4,96,happy
4,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Dua Lipa,Dance The Night,...,110.056,audio_features,1vYXt7VSjH9JIM5oRRo7vA,spotify:track:1vYXt7VSjH9JIM5oRRo7vA,https://api.spotify.com/v1/tracks/1vYXt7VSjH9J...,https://api.spotify.com/v1/audio-analysis/1vYX...,176579,4,95,happy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,152,"Interested, Excited","Outside - kazukii, Florence - fate , gasoline ...","Borat, borat subsequent movie film, Bruno, mis...","Electronic, Hip Hop, Pop, R&B, Alternative","Action, Comedy, Thriller",2,1,kazukii,Outside,...,99.981,audio_features,1dYo3EOe6Rw3cTT2OlZIF4,spotify:track:1dYo3EOe6Rw3cTT2OlZIF4,https://api.spotify.com/v1/tracks/1dYo3EOe6Rw3...,https://api.spotify.com/v1/audio-analysis/1dYo...,211200,4,49,happy
806,152,"Interested, Excited","Outside - kazukii, Florence - fate , gasoline ...","Borat, borat subsequent movie film, Bruno, mis...","Electronic, Hip Hop, Pop, R&B, Alternative","Action, Comedy, Thriller",2,1,fate,Florence,...,94.974,audio_features,3ZBqqfUUrFekZ8WwPQX4cu,spotify:track:3ZBqqfUUrFekZ8WwPQX4cu,https://api.spotify.com/v1/tracks/3ZBqqfUUrFek...,https://api.spotify.com/v1/audio-analysis/3ZBq...,166737,4,33,calm
807,152,"Interested, Excited","Outside - kazukii, Florence - fate , gasoline ...","Borat, borat subsequent movie film, Bruno, mis...","Electronic, Hip Hop, Pop, R&B, Alternative","Action, Comedy, Thriller",2,1,trap boy Freddy,gasoline,...,120.014,audio_features,1frGB2A7OcS7CgdH2C6apy,spotify:track:1frGB2A7OcS7CgdH2C6apy,https://api.spotify.com/v1/tracks/1frGB2A7OcS7...,https://api.spotify.com/v1/audio-analysis/1frG...,181812,4,49,sad
808,152,"Interested, Excited","Outside - kazukii, Florence - fate , gasoline ...","Borat, borat subsequent movie film, Bruno, mis...","Electronic, Hip Hop, Pop, R&B, Alternative","Action, Comedy, Thriller",2,1,NaN,rip Dutch Mel rose,...,146.834,audio_features,1bXbhCCiK70rhaqWQvDJvZ,spotify:track:1bXbhCCiK70rhaqWQvDJvZ,https://api.spotify.com/v1/tracks/1bXbhCCiK70r...,https://api.spotify.com/v1/audi

In [54]:
# Concatenate mood PROBABILITY predictions to main dataframe
df[['sad','happy','energetic','calm']] = loaded_model.predict_proba(X_transformed)
df

,Response_Id,Current_Mood,Recent_Songs,Recent_Movies,Music_Genre,Movie_Genre,Mood_to_Entrtnmnt,Music_to_Movie,Artist,Track,...,track_href,analysis_url,duration_ms,time_signature,popularity,mood,sad,happy,energetic,calm
0,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Lizzo,Pink,...,https://api.spotify.com/v1/tracks/2YXGbxICUdOU...,https://api.spotify.com/v1/audio-analysis/2YXG...,143833,4,76,happy,0.01,0.940000,0.050000,0.00
1,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Beyonce,Cuff It,...,https://api.spotify.com/v1/tracks/1xzi1Jcr7mEi...,https://api.spotify.com/v1/audio-analysis/1xzi...,225389,4,86,happy,0.00,0.800000,0.200000,0.00
2,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Olivia Rodrigo,Bad Idea Right,...,https://api.spotify.com/v1/tracks/3IX0yuEVvDbn...,https://api.spotify.com/v1/audio-analysis/3IX0...,184784,4,94,energetic,0.00,0.306667,0.693333,0.00
3,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Olivia Rodrigo,Vampire,...,https://api.spotify.com/v1/tracks/1kuGVB7EU95p...,https://api.spotify.com/v1/audio-analysis/1kuG...,219724,4,96,happy,0.26,0.660000,0.060000,0.02
4,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Dua Lipa,Dance The Night,...,https://api.spotify.com/v1/tracks/1vYXt7VSjH9J...,https://api.spotify.com/v1/audio-analysis/1vYX...,176579,4,95,happy,0.00,0.770000,0.230000,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,152,"Interested, Excited","Outside - kazukii, Florence - fate , gasoline ...","Borat, borat subsequent movie film, Bruno, mis...","Electronic, Hip Hop, Pop, R&B, Alternative","Action, Comedy, Thriller",2,1,kazukii,Outside,...,https://api.spotify.com/v1/tracks/1dYo3EOe6Rw3...,https://api.spotify.com/v1/audio-analysis/1dYo...,211200,4,49,happy,0.27,0.620000,0.090000,0.02
806,152,"Interested, Excited","Outside - kazukii, Florence - fate , gasoline ...","Borat, borat subsequent movie film, Bruno, mis...","Electronic, Hip Hop, Pop, R&B, Alternative","Action, Comedy, Thriller",2,1,fate,Florence,...,https://api.spotify.com/v1/tracks/3ZBqqfUUrFek...,https://api.spotify.com/v1/audio-analysis/3ZBq...,166737,4,33,calm,0.36,0.150000,0.070000,0.42
807,152,"Interested, Excited","Outside - kazukii, Florence - fate , gasoline ...","Borat, borat subsequent movie film, Bruno, mis...","Electronic, Hip Hop, Pop, R&B, Alternative","Action, Comedy, Thriller",2,1,trap boy Freddy,gasoline,...,https://api.spotify.com/v1/tracks/1frGB2A7OcS7...,https://api.spotify.com/v1/audio-analysis/1frG...,181812,4,49,sad,0.47,0.380000,0.140000,0.01
808,152,"Interested, Excited","Outside - kazukii, Florence - fate , gasoline ...","Borat, borat subsequent movie film, Bruno, mis...","Electronic, Hip Hop, Pop, R&B, Alternative","Action, Comedy, Thriller",2,1,NaN,rip Dutch Mel rose,...,https://api.spotify.com/v1/tracks/1bXbhCCiK70r...,https://api.spotify.com/v1/audio-analysis/1bXb...,142800,4,25,sad,0.51,0.460000,0.020000,0.01


## Average the outputted happy/sad/energetic/calm vectors

In [65]:
# Get data for one user
df_one_user = df[df['Response_Id']==0]
df_one_user

,Response_Id,Current_Mood,Recent_Songs,Recent_Movies,Music_Genre,Movie_Genre,Mood_to_Entrtnmnt,Music_to_Movie,Artist,Track,...,track_href,analysis_url,duration_ms,time_signature,popularity,mood,sad,happy,energetic,calm
0,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Lizzo,Pink,...,https://api.spotify.com/v1/tracks/2YXGbxICUdOU...,https://api.spotify.com/v1/audio-analysis/2YXG...,143833,4,76,happy,0.01,0.940000,0.050000,0.00
1,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Beyonce,Cuff It,...,https://api.spotify.com/v1/tracks/1xzi1Jcr7mEi...,https://api.spotify.com/v1/audio-analysis/1xzi...,225389,4,86,happy,0.00,0.800000,0.200000,0.00
2,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Olivia Rodrigo,Bad Idea Right,...,https://api.spotify.com/v1/tracks/3IX0yuEVvDbn...,https://api.spotify.com/v1/audio-analysis/3IX0...,184784,4,94,energetic,0.00,0.306667,0.693333,0.00
3,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Olivia Rodrigo,Vampire,...,https://api.spotify.com/v1/tracks/1kuGVB7EU95p...,https://api.spotify.com/v1/audio-analysis/1kuG...,219724,4,96,happy,0.26,0.660000,0.060000,0.02
4,0,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,3,Dua Lipa,Dance The Night,...,https://api.spotify.com/v1/tracks/1vYXt7VSjH9J...,https://api.spotify.com/v1/audio-analysis/1vYX...,176579,4,95,happy,0.00,0.770000,0.230000,0.00


In [66]:
user_mood_vector = df_one_user[['happy', 'sad', 'energetic', 'calm']].mean()
user_mood_vector

happy        0.695333
sad          0.054000
energetic    0.246667
calm         0.004000
dtype: float64

In [67]:
# Reshape mood vector for cosine similarity 
user_mood_reshaped = user_mood_vector.values.reshape(1, -1)

## Load movie data and perform cosine similarity to get top 5 movie recommendations

#### NOTE: Make sure you unzip file below before reading

In [63]:
movie_df = pd.read_csv('../data/raw/wiki_movies_mood_label_by_gpt-3.5-turbo_final.csv',',').loc[:,'year':]
movie_df

/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3553: FutureWarning: In a future version of pandas all arguments of read_csv except for the argument 'filepath_or_buffer' will be keyword-only
  exec(code_obj, self.user_global_ns, self.user_ns)


,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,based_on,starring,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
0,1973,40_Carats_(film),40 Carats (film),"Ann Stanley, who sells real estate in New York...",NaN,NaN,Directed byMilton Katselas,Screenplay byLeonard Gershe,Based on\nQuarante caratsby Pierre BarilletJea...,Starring\nLiv Ullmann\nEdward Albert\nGene Kel...,...,NaN,0,0,0,0,0.8,0.3,0.6,0.7,NaN
1,1973,Ace_Eli_and_Rodger_of_the_Skies,Ace Eli and Rodger of the Skies,"In the early 1920s, Eli (Cliff Robertson) is a...",NaN,NaN,Directed byJohn Erman,Screenplay byClaudia Salter,NaN,Starring\nCliff Robertson\nEric Shea\nPamela F...,...,NaN,0,0,0,0,0.7,0.6,0.8,0.5,NaN
2,1973,The_Affair_(1973_film),The Affair (1973 film),Courtney Patterson is a beautiful 32 year old ...,NaN,NaN,Directed byGilbert Cates,NaN,NaN,StarringNatalie WoodRobert WagnerBruce Davison...,...,NaN,0,0,0,0,0.5,0.8,0.4,0.7,NaN
3,1973,The_All-American_Boy_(film),The All-American Boy (film),"Vic Bealer, a young boxer from a small town in...",NaN,NaN,Directed byCharles Eastman,NaN,NaN,StarringJon VoightE. J. Peaker,...,NaN,0,0,0,0,0.2,0.6,0.8,0.4,NaN
4,1973,American_Graffiti,American Graffiti,On their last evening of summer vacation in 19...,NaN,NaN,Directed byGeorge Lucas,NaN,NaN,Starring\nRichard Dreyfuss\nRon Howard\nPaul L...,...,180.18018,1,2,1,2,0.5,0.7,0.7,0.5,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11933,2023,Rebel_Moon,Rebel Moon,NaN,In a universe controlled by the corrupt govern...,NaN,Directed byZack Snyder,Screenplay by\nZack Snyder\nKurt Johnstad\nSha...,NaN,Starring\nSofia Boutella\nCharlie Hunnam\nMich...,...,NaN,0,0,0,0,0.3,0.4,0.8,0.5,NaN
11934,2023,The_Iron_Claw_(film),The Iron Claw (film),NaN,NaN,The film is centered around the Von Erich fami...,Directed bySean Durkin,NaN,NaN,Starring\nZac Efron\nJeremy Allen White\nHarri...,...,NaN,0,0,0,0,0.3,0.7,0.8,0.2,NaN
11935,2023,The_Color_Purple_(2023_film),The Color Purple (2023 film),NaN,A story of the life-long struggles of an Afric...,NaN,Directed byBlitz Bazawule,Screenplay byMarcus Gardley,Based on\nThe Color Purpleby Alice Walker\nThe...,Starring\nTaraji P. Henson\nDanielle Brooks\nC...,...,NaN,0,0,0,0,0.1,0.9,0.2,0.8,NaN
11936,2023,The_Boys_in_the_Boat_(film),The Boys in the Boat (film),The non-fiction novel describes the University...,NaN,NaN,Directed byGeorge Clooney,Screenplay byMark L. Smith,Based onThe Boys in the Boatby Daniel James Brown,Starring\nCallum Turner\nJoel Edgerton\n,...,NaN,0,0,0,0,0.2,0.7,0.8,0.4,NaN


In [64]:
# Load mood vectors for movies
movie_mood_vectors = movie_df[['happy','sad','energetic','calm']].values
movie_mood_vectors

array([[0.8, 0.3, 0.6, 0.7],
       [0.7, 0.6, 0.8, 0.5],
       [0.5, 0.8, 0.4, 0.7],
       ...,
       [0.1, 0.9, 0.2, 0.8],
       [0.2, 0.7, 0.8, 0.4],
       [0.7, 0.3, 0.9, 0.4]])

### Simple cosine similarity

In [79]:
# Perform cosine similarity to get top 5 similar movies based on mood vector
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(user_mood_reshaped, movie_mood_vectors)
sim_scores = list(enumerate(similarities[0]))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

# Get top movie titles based on top similarity scores
top_movies = sim_scores[1:6]
top_movie_indices = [i[0] for i in top_movies]
recommended_movies = movie_df.iloc[top_movie_indices]
recommended_movies

,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,based_on,starring,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
2458,1987,Cannon_Movie_Tales,Cannon Movie Tales,NaN,NaN,A childless Queen (Morgan Fairchild) and her K...,NaN,NaN,NaN,NaN,...,NaN,0,0,0,0,1.0,0.2,0.7,0.1,NaN
8246,2009,My_Little_Pony:_A_Very_Pony_Place,My Little Pony: A Very Pony Place,"Unlike the previous three specials, A Very Pon...",NaN,NaN,Directed byJohn Grusd,NaN,Based onMy Little Pony by Bonnie Zacherle,StarringKathleen BarrAdrienne CarterAnna Cumme...,...,NaN,0,0,0,0,0.8,0.1,0.5,0.2,NaN
695,1977,Greased_Lightning_(1977_film),Greased Lightning (1977 film),"In 1930s Danville, Virginia, an African-Americ...",NaN,NaN,Directed byMichael Schultz,NaN,NaN,StarringRichard PryorBeau BridgesPam GrierClea...,...,NaN,0,0,0,0,1.0,0.2,0.7,0.2,NaN
5612,1998,Meet_the_Deedles,Meet the Deedles,Fraternal twin brothers Phil (Walker) and Stew...,NaN,NaN,Directed bySteve Boyum,NaN,NaN,Starring\nPaul Walker\nSteve Van Wormer\nJohn ...,...,0.183333,0,0,0,0,0.9,0.1,0.7,0.2,NaN
4411,1994,Richie_Rich_(film),Richie Rich (film),"Richard ""Richie"" Rich, Jr. is ""the world's ric...",NaN,NaN,Directed byDonald Petrie,Screenplay byTom S. ParkerJim Jennewein,Based onRichie Richby Alfred HarveyWarren Kremer,Starring\nMacaulay Culkin\nJohn Larroquette\nE...,...,1.900000,1,1,0,0,1.0,0.1,0.8,0.2,NaN


### Alternate 2: Cosine similarity + popularity filter + after 1995

In [77]:
similarities = cosine_similarity(user_mood_reshaped, movie_mood_vectors)
sim_scores = list(enumerate(similarities[0]))

# Filter movies released in 1995 or after
eligible_movies = [i for i in sim_scores if movie_df.loc[i[0], 'year'] >= 1995]

# Sort by cosine similarity and popularity score
eligible_movies = sorted(eligible_movies, key=lambda x: (x[1], movie_df.loc[x[0], 'rotten_tomatoes_score']), reverse=True)

# Filter out NaN values and select top movies
top_movies = [i for i in eligible_movies if not pd.isna(movie_df.loc[i[0], 'rotten_tomatoes_score']) or not pd.isna(movie_df.loc[i[0], 'metacritic_score'])][:5]
top_movie_indices = [i[0] for i in top_movies]
recommended_movies = movie_df.iloc[top_movie_indices]
recommended_movies

,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,based_on,starring,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
5612,1998,Meet_the_Deedles,Meet the Deedles,Fraternal twin brothers Phil (Walker) and Stew...,NaN,NaN,Directed bySteve Boyum,NaN,NaN,Starring\nPaul Walker\nSteve Van Wormer\nJohn ...,...,0.183333,0,0,0,0,0.9,0.1,0.7,0.2,NaN
8064,2009,Bride_Wars,Bride Wars,"Emma Allan and Olivia ""Liv"" Lerner are childho...",NaN,NaN,Directed byGary Winick,Screenplay by\nGreg DePaul\nCasey Wilson\nJune...,NaN,Starring\nKate Hudson\nAnne Hathaway\nSteve Ho...,...,3.846667,1,2,0,0,1.0,0.1,0.8,0.2,NaN
7753,2007,Enchanted_(film),Enchanted (film),In the animated fairy tale kingdom of Andalasi...,NaN,NaN,Directed byKevin Lima,NaN,NaN,Starring\nAmy Adams\nPatrick Dempsey\nJames Ma...,...,4.005882,1,2,1,0,1.0,0.2,0.7,0.3,NaN
5044,1996,Race_the_Sun_(film),Race the Sun (film),"In Hawaii, Sandra Beecher arrives at Kona Pali...",NaN,NaN,Directed byCharles T. Kanganis,NaN,NaN,Starring\nHalle Berry\nCasey Affleck\nEliza Du...,...,NaN,0,0,0,0,0.9,0.2,0.8,0.1,NaN
7589,2007,Blades_of_Glory,Blades of Glory,"At the 2002 World Winter Sport Games, raunchy ...",NaN,NaN,Directed by\nWill SpeckJosh Gordon\n,Screenplay by\nJeff Cox\nCraig Cox\nJohn Altsc...,NaN,Starring\nWill Ferrell\nJon Heder\nWill Arnett...,...,2.388525,1,1,1,0,1.0,0.2,0.9,0.1,NaN


### Alternate 3: Cosine similarity + popularity filter

In [78]:
similarities = cosine_similarity(user_mood_reshaped, movie_mood_vectors)
sim_scores = list(enumerate(similarities[0]))


# Sort by cosine similarity and popularity score
eligible_movies = sorted(sim_scores, key=lambda x: (x[1], movie_df.loc[x[0], 'rotten_tomatoes_score']), reverse=True)

# Filter out NaN values and select top movies
top_movies = [i for i in eligible_movies if not pd.isna(movie_df.loc[i[0], 'rotten_tomatoes_score']) or not pd.isna(movie_df.loc[i[0], 'metacritic_score'])][:5]
top_movie_indices = [i[0] for i in top_movies]
recommended_movies = movie_df.iloc[top_movie_indices]
recommended_movies

,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,based_on,starring,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
695,1977,Greased_Lightning_(1977_film),Greased Lightning (1977 film),"In 1930s Danville, Virginia, an African-Americ...",NaN,NaN,Directed byMichael Schultz,NaN,NaN,StarringRichard PryorBeau BridgesPam GrierClea...,...,NaN,0,0,0,0,1.0,0.2,0.7,0.2,NaN
5612,1998,Meet_the_Deedles,Meet the Deedles,Fraternal twin brothers Phil (Walker) and Stew...,NaN,NaN,Directed bySteve Boyum,NaN,NaN,Starring\nPaul Walker\nSteve Van Wormer\nJohn ...,...,0.183333,0,0,0,0,0.9,0.1,0.7,0.2,NaN
4411,1994,Richie_Rich_(film),Richie Rich (film),"Richard ""Richie"" Rich, Jr. is ""the world's ric...",NaN,NaN,Directed byDonald Petrie,Screenplay byTom S. ParkerJim Jennewein,Based onRichie Richby Alfred HarveyWarren Kremer,Starring\nMacaulay Culkin\nJohn Larroquette\nE...,...,1.900000,1,1,0,0,1.0,0.1,0.8,0.2,NaN
8064,2009,Bride_Wars,Bride Wars,"Emma Allan and Olivia ""Liv"" Lerner are childho...",NaN,NaN,Directed byGary Winick,Screenplay by\nGreg DePaul\nCasey Wilson\nJune...,NaN,Starring\nKate Hudson\nAnne Hathaway\nSteve Ho...,...,3.846667,1,2,0,0,1.0,0.1,0.8,0.2,NaN
7753,2007,Enchanted_(film),Enchanted (film),In the animated fairy tale kingdom of Andalasi...,NaN,NaN,Directed byKevin Lima,NaN,NaN,Starring\nAmy Adams\nPatrick Dempsey\nJames Ma...,...,4.005882,1,2,1,0,1.0,0.2,0.7,0.3,NaN
